In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

conf = SparkConf()
conf.set('spark.app.name', 'PySpark SQL Join')
conf.set('spark.master', 'local[*]')

spark = SparkSession.builder\
        .config(conf = conf)\
        .getOrCreate()

In [2]:
vital = [
     { 'UserID': 100, 'VitalID': 1, 'Date': '2020-01-01', 'Weight': 75 },
     { 'UserID': 100, 'VitalID': 2, 'Date': '2020-01-02', 'Weight': 78 },
     { 'UserID': 101, 'VitalID': 3, 'Date': '2020-01-01', 'Weight': 90 },
     { 'UserID': 101, 'VitalID': 4, 'Date': '2020-01-02', 'Weight': 95 },
]

alert = [
    { 'AlertID': 1, 'VitalID': 4, 'AlertType': 'WeightIncrease', 'Date': '2020-01-01', 'UserID': 101},
    { 'AlertID': 2, 'VitalID': None, 'AlertType': 'MissingVital', 'Date': '2020-01-04', 'UserID': 100},
    { 'AlertID': 3, 'VitalID': None, 'AlertType': 'MissingVital', 'Date': '2020-01-05', 'UserID': 101}
]

vital_df = spark.createDataFrame(vital)
alert_df = spark.createDataFrame(alert)

In [3]:
vital_df.printSchema()
alert_df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- UserID: long (nullable = true)
 |-- VitalID: long (nullable = true)
 |-- Weight: long (nullable = true)

root
 |-- AlertID: long (nullable = true)
 |-- AlertType: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- UserID: long (nullable = true)
 |-- VitalID: long (nullable = true)



In [4]:
vital_df.show()
alert_df.show()

+----------+------+-------+------+
|      Date|UserID|VitalID|Weight|
+----------+------+-------+------+
|2020-01-01|   100|      1|    75|
|2020-01-02|   100|      2|    78|
|2020-01-01|   101|      3|    90|
|2020-01-02|   101|      4|    95|
+----------+------+-------+------+

+-------+--------------+----------+------+-------+
|AlertID|     AlertType|      Date|UserID|VitalID|
+-------+--------------+----------+------+-------+
|      1|WeightIncrease|2020-01-01|   101|      4|
|      2|  MissingVital|2020-01-04|   100|   NULL|
|      3|  MissingVital|2020-01-05|   101|   NULL|
+-------+--------------+----------+------+-------+



In [5]:
vital_df.createOrReplaceTempView('vital')
alert_df.createOrReplaceTempView('alert')

### JOIN

1. **INNER JOIN**

In [16]:
spark.sql("""
    SELECT *
    FROM vital v
    JOIN alert a ON v.VitalID = a.VitalID
""").show()

+----------+------+-------+------+-------+--------------+----------+------+-------+
|      Date|UserID|VitalID|Weight|AlertID|     AlertType|      Date|UserID|VitalID|
+----------+------+-------+------+-------+--------------+----------+------+-------+
|2020-01-02|   101|      4|    95|      1|WeightIncrease|2020-01-01|   101|      4|
+----------+------+-------+------+-------+--------------+----------+------+-------+



2. **OUTER JOIN**
- LEFT OUTER JOIN
- RIGHT OUTER JOIN
- FULL OUTER JOIN

In [17]:
spark.sql("""
    SELECT *
    FROM vital v
    LEFT JOIN alert a ON v.VitalID = a.VitalID       
""").show()

+----------+------+-------+------+-------+--------------+----------+------+-------+
|      Date|UserID|VitalID|Weight|AlertID|     AlertType|      Date|UserID|VitalID|
+----------+------+-------+------+-------+--------------+----------+------+-------+
|2020-01-01|   100|      1|    75|   NULL|          NULL|      NULL|  NULL|   NULL|
|2020-01-02|   100|      2|    78|   NULL|          NULL|      NULL|  NULL|   NULL|
|2020-01-01|   101|      3|    90|   NULL|          NULL|      NULL|  NULL|   NULL|
|2020-01-02|   101|      4|    95|      1|WeightIncrease|2020-01-01|   101|      4|
+----------+------+-------+------+-------+--------------+----------+------+-------+



In [18]:
spark.sql("""
    SELECT *
    FROM vital v
    RIGHT JOIN alert a ON v.VitalID = a.VitalID       
""").show()

+----------+------+-------+------+-------+--------------+----------+------+-------+
|      Date|UserID|VitalID|Weight|AlertID|     AlertType|      Date|UserID|VitalID|
+----------+------+-------+------+-------+--------------+----------+------+-------+
|2020-01-02|   101|      4|    95|      1|WeightIncrease|2020-01-01|   101|      4|
|      NULL|  NULL|   NULL|  NULL|      2|  MissingVital|2020-01-04|   100|   NULL|
|      NULL|  NULL|   NULL|  NULL|      3|  MissingVital|2020-01-05|   101|   NULL|
+----------+------+-------+------+-------+--------------+----------+------+-------+



In [19]:
spark.sql("""
    SELECT *
    FROM vital v
    FULL JOIN alert a ON v.VitalID = a.VitalID       
""").show()

+----------+------+-------+------+-------+--------------+----------+------+-------+
|      Date|UserID|VitalID|Weight|AlertID|     AlertType|      Date|UserID|VitalID|
+----------+------+-------+------+-------+--------------+----------+------+-------+
|      NULL|  NULL|   NULL|  NULL|      2|  MissingVital|2020-01-04|   100|   NULL|
|      NULL|  NULL|   NULL|  NULL|      3|  MissingVital|2020-01-05|   101|   NULL|
|2020-01-01|   100|      1|    75|   NULL|          NULL|      NULL|  NULL|   NULL|
|2020-01-02|   100|      2|    78|   NULL|          NULL|      NULL|  NULL|   NULL|
|2020-01-01|   101|      3|    90|   NULL|          NULL|      NULL|  NULL|   NULL|
|2020-01-02|   101|      4|    95|      1|WeightIncrease|2020-01-01|   101|      4|
+----------+------+-------+------+-------+--------------+----------+------+-------+



3. **CROSS JOIN**

In [6]:
spark.sql("""
    SELECT *
    FROM vital CROSS JOIN alert
""").show()

+----------+------+-------+------+-------+--------------+----------+------+-------+
|      Date|UserID|VitalID|Weight|AlertID|     AlertType|      Date|UserID|VitalID|
+----------+------+-------+------+-------+--------------+----------+------+-------+
|2020-01-01|   100|      1|    75|      1|WeightIncrease|2020-01-01|   101|      4|
|2020-01-01|   100|      1|    75|      2|  MissingVital|2020-01-04|   100|   NULL|
|2020-01-01|   100|      1|    75|      3|  MissingVital|2020-01-05|   101|   NULL|
|2020-01-02|   100|      2|    78|      1|WeightIncrease|2020-01-01|   101|      4|
|2020-01-02|   100|      2|    78|      2|  MissingVital|2020-01-04|   100|   NULL|
|2020-01-02|   100|      2|    78|      3|  MissingVital|2020-01-05|   101|   NULL|
|2020-01-01|   101|      3|    90|      1|WeightIncrease|2020-01-01|   101|      4|
|2020-01-01|   101|      3|    90|      2|  MissingVital|2020-01-04|   100|   NULL|
|2020-01-01|   101|      3|    90|      3|  MissingVital|2020-01-05|   101| 

4. **SELF JOIN**

In [7]:
spark.sql("""
    SELECT *
    FROM vital JOIN vital
""").show()

+----------+------+-------+------+----------+------+-------+------+
|      Date|UserID|VitalID|Weight|      Date|UserID|VitalID|Weight|
+----------+------+-------+------+----------+------+-------+------+
|2020-01-01|   100|      1|    75|2020-01-01|   100|      1|    75|
|2020-01-01|   100|      1|    75|2020-01-02|   100|      2|    78|
|2020-01-01|   100|      1|    75|2020-01-01|   101|      3|    90|
|2020-01-01|   100|      1|    75|2020-01-02|   101|      4|    95|
|2020-01-02|   100|      2|    78|2020-01-01|   100|      1|    75|
|2020-01-02|   100|      2|    78|2020-01-02|   100|      2|    78|
|2020-01-02|   100|      2|    78|2020-01-01|   101|      3|    90|
|2020-01-02|   100|      2|    78|2020-01-02|   101|      4|    95|
|2020-01-01|   101|      3|    90|2020-01-01|   100|      1|    75|
|2020-01-01|   101|      3|    90|2020-01-02|   100|      2|    78|
|2020-01-01|   101|      3|    90|2020-01-01|   101|      3|    90|
|2020-01-01|   101|      3|    90|2020-01-02|   

In [8]:
spark.sql("""
    SELECT *
    FROM vital v1 JOIN vital v2 ON v1.VitalID = v2.VitalID
""").show()

+----------+------+-------+------+----------+------+-------+------+
|      Date|UserID|VitalID|Weight|      Date|UserID|VitalID|Weight|
+----------+------+-------+------+----------+------+-------+------+
|2020-01-01|   100|      1|    75|2020-01-01|   100|      1|    75|
|2020-01-02|   100|      2|    78|2020-01-02|   100|      2|    78|
|2020-01-01|   101|      3|    90|2020-01-01|   101|      3|    90|
|2020-01-02|   101|      4|    95|2020-01-02|   101|      4|    95|
+----------+------+-------+------+----------+------+-------+------+

